In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from utils import CustomDataset,SelfAttention
import tiktoken

# Load variables and text

In [5]:
enc = tiktoken.get_encoding("cl100k_base")
print(f"{enc.n_vocab} tokens in the encoding")

with open("sample_text.txt", "r") as f:
    text = f.read()

enc_text = enc.encode(text)
print(f"Encoded text length: {len(enc_text)}")

100277 tokens in the encoding
Encoded text length: 304


# Code from self-attention notebook

In [30]:
dataloader = DataLoader(CustomDataset(text, enc, max_length=4, stride=4), batch_size=1, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
input_ids, target_ids = first_batch


vicab_size = enc.n_vocab
print(f"Vocab size: {vicab_size}") 

embed_dims = 8 
print(f"Output dims: {embed_dims}")

token_embed = torch.nn.Embedding(vicab_size, embed_dims)
print(f"Token embedding shape: {token_embed.weight.shape}")

inputs = token_embed(input_ids)
print(f"Input shape: {inputs.shape} -> batch_size x sequence_length x embedding_dim")

d_in = inputs.shape[-1]
d_out = 8 # dimensions of the output 

Vocab size: 100277
Output dims: 8
Token embedding shape: torch.Size([100277, 8])
Input shape: torch.Size([1, 4, 8]) -> batch_size x sequence_length x embedding_dim


In [21]:
Query_w = torch.nn.Linear(d_in, d_out, bias=False)
Key_w = torch.nn.Linear(d_in, d_out, bias=False)
Value_w = torch.nn.Linear(d_in, d_out, bias=False)
print(f"Query weight shape: {Query_w.weight.shape} -> input_dim x output_dim")
print(f"Key weight shape: {Key_w.weight.shape} -> input_dim x output_dim")
print(f"Value weight shape: {Value_w.weight.shape} -> input_dim x output_dim")

Query weight shape: torch.Size([8, 8]) -> input_dim x output_dim
Key weight shape: torch.Size([8, 8]) -> input_dim x output_dim
Value weight shape: torch.Size([8, 8]) -> input_dim x output_dim


In [23]:
query = Query_w(inputs) # batch_size x sequence_length x d_out
key = Key_w(inputs) # batch_size x sequence_length x d_out
value = Value_w(inputs) # batch_size x sequence_length x d_out

attention_scores = query @ key.transpose(1, 2) / (d_out ** 0.5)

In [24]:
attention_scores

tensor([[[ 0.7236,  0.1185,  0.8006,  0.2192],
         [-0.2173, -0.0825, -0.3061, -0.1011],
         [ 0.2715,  0.1823,  0.5676,  0.2847],
         [ 0.2003,  0.1276,  0.2629,  0.0794]]], grad_fn=<DivBackward0>)

# Masking 
- use -inf for upper triangle
- use 0 for lower triangle

In [12]:
mask_upper = torch.triu(torch.ones(attention_scores.shape[1], attention_scores.shape[1]), diagonal=1)
mask_upper

tensor([[0., 1., 1., 1.],
        [0., 0., 1., 1.],
        [0., 0., 0., 1.],
        [0., 0., 0., 0.]])

# Fill with -inf where mask is 1 

In [13]:
masked_attention_scores = attention_scores.masked_fill(mask_upper == 1, float("-inf"))
masked_attention_scores

tensor([[[-0.4336,    -inf,    -inf,    -inf],
         [ 0.0551, -0.5242,    -inf,    -inf],
         [-0.2982, -0.1188, -0.2528,    -inf],
         [-0.0330,  0.0750, -0.1073,  0.1535]]], grad_fn=<MaskedFillBackward0>)

# New attention weights

In [15]:
attention_weights = torch.nn.functional.softmax(masked_attention_scores, dim=-1)
attention_weights

tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.6409, 0.3591, 0.0000, 0.0000],
         [0.3084, 0.3690, 0.3227, 0.0000],
         [0.2354, 0.2623, 0.2186, 0.2837]]], grad_fn=<SoftmaxBackward0>)

# Masked attention - Causal Attention

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in,d_out,context_length,dropout=0.1):
        super(CausalAttention, self).__init__()
        """
        d_in : embedding dimension at the input
        d_out : embedding dimension at the output = head_dims 
        context_length : length of the input sequence
        
        Query_w : Linear layer for of shape (d_in, d_out) 
        
        query : batch_size x sequence_length x d_out
        """
        
        self.query_w = nn.Linear(d_in, d_out, bias=False)
        self.key_w = nn.Linear(d_in, d_out, bias=False)
        self.value_w = nn.Linear(d_in, d_out, bias=False)
        self.d_out = d_out
        self.dropout = nn.Dropout(0.1)
        self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
    
    def forward(self, x):
        query = self.query_w(x) # same as x @ Query_w.weight.T + Query_w.bias (none)
        key = self.key_w(x) # same as x @ Key_w.weight.T + Key_w.bias (none)
        value = self.value_w(x) # same as x @ Value_w.weight.T + Value_w.bias (none)

        attention_scores = query @ key.transpose(1, 2) / (self.d_out ** 0.5)
        masked_attention_scores = attention_scores.masked_fill(self.mask == 1, float("-inf"))
        attention_weights = torch.nn.functional.softmax(masked_attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        return attention_weights @ value # context vector

In [29]:
context_length = inputs.shape[1]
causal_attention = CausalAttention(d_in, d_out, context_length=context_length)
causal_attention(inputs)

tensor([[[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
           0.0000],
         [ 0.5620, -0.3022, -0.2023,  0.7372,  0.1131, -0.5299,  0.0033,
          -0.5410],
         [ 0.1561, -0.5996,  0.1581,  0.3086,  0.0628, -0.1369,  0.2652,
          -0.1809],
         [ 0.1290, -0.4186,  0.0797,  0.0460,  0.1589, -0.1599,  0.0557,
          -0.1484]]], grad_fn=<UnsafeViewBackward0>)